# Decorators: The Sandwich Story

A **decorator** is a function that takes a function as an argument and returns a new function — wrapping it with extra behaviour.

We'll build up from manual wrapping to full decorator factories, using sandwiches as our metaphor.

In [2]:
# Step 1: Manual wrapping — the problem decorators solve
#
# We want every filling to be wrapped in bread.
# Without decorators, we have to wrap manually each time.

def bread(filling_func):
    def sandwich():
        print("<------>")
        filling_func()
        print("<\\____/>")
    return sandwich

def filler():
    print("--~~~~~~~--")

filler = bread(filler)  # manual wrap — verbose and easy to forget
filler()

<------>
--~~~~~~~--
<\____/>


## Step 2: The `@` Decorator Syntax

Python's `@` syntax is shorthand for the manual wrapping above:

```python
@bread
def filler():
    ...
```

is exactly the same as:

```python
def filler():
    ...
filler = bread(filler)
```

In [4]:
# Step 2: Using @ decorator syntax
# @bread is equivalent to: filler = bread(filler)

def bread(filling_func):
    def sandwich():
        print("<------>")
        filling_func()
        print("<\\____/>")
    return sandwich

@bread
def filler():
    print("--~~~~~~~--")

filler()

<------>
--~~~~~~~--
<\____/>


## Step 3: Passing Arguments Through

The inner `sandwich` function must accept and forward any arguments the original function expects.

In [5]:
# Step 3: Decorator that passes arguments to the wrapped function

def bread(filling_func):
    def sandwich(ingredient):        # accept the argument
        print("<------\\>")
        filling_func(ingredient)     # forward it
        print("<\\____/>")
    return sandwich

@bread
def filler(ingredient):
    print(f"--{ingredient}--")

filler("ham")
print()
filler("cheese")

<------\>
--ham--
<\____/>

<------\>
--cheese--
<\____/>


In [6]:
filler("tofu")

<------\>
--tofu--
<\____/>


## Step 4: Stacking Decorators

Decorators can be stacked. They are applied **bottom-up** at definition time, but execute **top-down** at call time:

```python
@bread          # applied second → outermost layer
@salad          # applied first  → inner layer
def filler(): ...
```

is equivalent to `filler = bread(salad(filler))`.

In [7]:
# Step 4: Stacking decorators — applied bottom-up, executed top-down

def bread(filling_func):
    def sandwich(ingredient):
        print("</------\\>")
        filling_func(ingredient)
        print("<\\____/>")
    return sandwich

def salad(filling_func):
    def sandwich(ingredient):
        print(" #######  ")
        filling_func(ingredient)
        print("  ~~~~~~  ")
    return sandwich

@bread          # outermost — executes first
@salad          # innermost — executes second
def filler(ingredient):
    print(f"--{ingredient}--")

# bread(salad(filler))("chicken")  # equivalent
filler("chicken")
print()
filler("veggie")

</------\>
 #######  
--chicken--
  ~~~~~~  
<\____/>

</------\>
 #######  
--veggie--
  ~~~~~~  
<\____/>


## Step 5: Decorator Factories

A **decorator factory** is a function that *returns* a decorator. This lets you pass configuration to the decorator itself:

```python
@bread("sourdough")   # bread("sourdough") returns a decorator, which then wraps filler
def filler(): ...
```

Three levels of nesting: factory → decorator → wrapper.

In [9]:
# Step 5: Decorator factory — a function that returns a decorator

def bread(crust):                       # factory: receives config
    def decorator(filling_func):        # decorator: receives the function
        def sandwich(*args, **kwargs):  # wrapper: receives the call arguments
            print(f"<== {crust} ==>")
            filling_func(*args, **kwargs)
            print(f"<===========>")
        return sandwich
    return decorator

#@tool(schema=mysupercoolschema)
@bread("sourdough")
def filler(ingredient):
    print(f"--{ingredient}--")

@bread("whole wheat")
def healthy_filler(ingredient):
    print(f"--{ingredient}--")

filler("avocado")
print()
healthy_filler("avocado")

<== sourdough ==>
--avocado--
<===========>

<== whole wheat ==>
--avocado--
<===========>
